# 04. Human API Deep Dive — How to Convert IDs and Interpret Results

*Last updated:* 2026-01-08

This notebook is a **hands-on tutorial** of the public IDTrack API using **human only**.
It is written to be both beginner-friendly and deep enough for advanced users.

You will learn:
1. how to initialize the human graph snapshot
2. how to convert identifiers one-by-one
3. how to convert identifiers in batches (and summarize outcomes)
4. how to understand ambiguity (1→n) and failures (1→0)
5. how to request a conversion explanation path


## 1. Setup and graph initialization

If you have already built the human graph in `initialization_graph.ipynb`, this step should mostly *load*
from cache.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

organism, latest_release = api.resolve_organism('human')
SNAPSHOT_RELEASE = latest_release

api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=True)
print('Ready:', organism, 'snapshot', SNAPSHOT_RELEASE)


## 2. The core function: `api.convert_identifier(...)`

This is the beginner-friendly entry point. It returns a dictionary with:
- the resolved internal graph ID (`graph_id`)
- the target IDs (`target_id`)
- flags that tell you what happened (`no_corresponding`, `no_conversion`, `no_target`)

### 2.1 Example: convert a gene symbol to the graph’s snapshot release


In [ ]:
api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE)


If you see `no_corresponding=True`, it means the input could not be matched.
Try a different spelling/casing, or use an Ensembl ID directly.


### 2.2 Example: time travel (convert to an older release)

Why this matters: published datasets often use older releases.


In [ ]:
# Choose an older release to demonstrate time travel
older_release = SNAPSHOT_RELEASE - 10
api.convert_identifier('TP53', to_release=older_release)


### 2.3 Convert into an external database (HGNC)

To convert into a specific external database, pass `final_database=...`.
Database names are the same names you see in your external YAML.


In [ ]:
api.convert_identifier('ENSG00000141510', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol')


### 2.4 How do I know which external databases are available?

Use the graph itself to list databases currently represented.


In [ ]:
g = api.track.graph
sorted(g.available_external_databases)[:50]


## 3. Understanding the result dictionary

Key fields:
- `query_id`: exactly what you typed
- `graph_id`: what IDTrack matched internally (normalization step)
- `target_id`: list of outputs (can be 0, 1, or many)
- `no_corresponding`: input didn’t match any node
- `no_conversion`: input matched, but no path to target release / database
- `no_target`: reached an Ensembl target, but requested external DB had no synonym

Important: `target_id` is a list because ambiguity is real and common.


## 4. Ambiguity control: `strategy='best'` vs `strategy='all'`

- `strategy='best'` (default): returns a single best target when possible
- `strategy='all'`: returns *all* candidates IDTrack found

Use `'all'` when you are doing QC or want to inspect ambiguous mappings.


In [ ]:
api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE, strategy='all')


## 5. Batch conversion (what you will do in real projects)

Most workflows start from a list of identifiers (genes in a count matrix, markers, hits, etc.).
IDTrack provides two helpers:
- `convert_identifier_multiple(...)`
- `classify_multiple_conversion(...)` to summarize outcomes


In [ ]:
genes = ['TP53', 'BRCA1', 'BRCA2', 'BRAF', 'KRAS', 'NOT_A_REAL_GENE']
results = api.convert_identifier_multiple(genes, to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol')
results[:2]  # show first two


In [ ]:
summary = api.classify_multiple_conversion(results)
# Each bin is a list of per-gene dictionaries
{k: len(v) for k, v in summary.items()}


If you want a human-readable report, you can print the summary bins:


In [ ]:
api.print_binned_conversion(summary)


## 6. Ask for an explanation path (`explain=True`)

When you set `explain=True`, the result includes a `the_path` field describing the graph edges followed.
This is very useful for advanced QC and debugging.


In [ ]:
explained = api.convert_identifier('TP53', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol', explain=True)
list(explained.keys())


In [ ]:
# The path dictionary keys are (target_id, ensembl_gene_id) pairs
list(explained['the_path'].keys())[:3]


`the_path` is intentionally detailed. For most users, the summary flags and `target_id` are enough.


## 7. Advanced: direct access to `Track` (power users)

`api.convert_identifier(...)` is a convenience wrapper around `api.track.convert(...)`.
If you need full control (search settings, path returns, scoring), you can call `Track.convert` directly.

Example (advanced):
```python
api.track.convert(
    from_id='TP53',
    from_release=None,
    to_release=SNAPSHOT_RELEASE,
    final_database='HGNC Symbol',
    return_path=True,
)
```


## 8. Practical advice (the kind that saves you a week)

1. Always record your **snapshot release** in your analysis notes.
2. If you share results, share the **external YAML** too.
3. When mapping is ambiguous, do not hide it — decide how your pipeline should handle 1→n mappings.
4. For scRNA-seq harmonization, prefer stable namespaces (Ensembl IDs) before switching to symbols.
